In [6]:
from itertools import combinations

import re
import unicodedata
import json

import pandas as pd

In [7]:

root_dir = '/home/bsc/bsc093754/GIT/social-media-data-map/data/processed/'
input_file = f'{root_dir}participant_data_id_merged.csv'
output_file = f'{root_dir}avg_pairwise_sim_results.json'

In [8]:


def standardize_strings(df, compare_column):
    """
    Standardize a string by:
    - converting to lowercase
    - removing accents
    - stripping leading/trailing whitespace
    - replacing multiple whitespace with a single space
    - removing punctuation (except letters, numbers, and spaces)
    """

    for ix, row in df.iterrows():

        text = row[compare_column]

        if text is None:
            return ""

        text = str(text)

        # Lowercase
        text = text.lower()

        # Remove accents
        text = unicodedata.normalize("NFKD", text)
        text = "".join(c for c in text if not unicodedata.combining(c))

        # Remove punctuation
        text = re.sub(r"[^\w\s]", "", text)

        # Normalize whitespace
        text = re.sub(r"\s+", " ", text).strip()

        df.at[ix, 'KeepIdStandard'] = text

    return df

 



In [9]:
def jaccard_similarity(a, b):
    set_a = set(a)
    set_b = set(b)
    # intersection of two sets
    intersection = len(set_a.intersection(set_b))
    # Unions of two sets
    union = len(set_a.union(set_b))
    
    return intersection / union

def dice_coefficient(a, b):
    """
    Compute the Dice coefficient between two collections.

    Parameters
    ----------
    a, b : iterable
        Lists, sets, tuples, etc.

    Returns
    -------
    float
        Dice coefficient in [0, 1].
    """
    set_a = set(a)
    set_b = set(b)

    if not set_a and not set_b:
        return 1.0

    intersection = len(set_a & set_b)

    return (2 * intersection) / (len(set_a) + len(set_b))



In [10]:
def avg_pair_sim(input_file, compare_column):

    df = pd.read_csv(input_file)
    df = standardize_strings(df, compare_column)

    platforms = df['platform'].unique()
    results_list = []

    for p in platforms:

        df_filtered = df[df['platform'] == p]

        js_total = 0
        dc_total = 0
        total = len(df )-1
        
        for (i1, row1), (i2, row2) in combinations(df_filtered.iterrows(), 2):

    
            value1 =row1['KeepIdStandard']
            value2 = row2['KeepIdStandard']

            js = jaccard_similarity(value1, value2)
            js_total = js_total + js

            dc = dice_coefficient(value1, value2)
            dc_total = dc_total + dc

        avg_js = js_total/total
        avg_dc = dc_total/total
        node = {'platform' : p, 
        'avg_js': avg_js,
        'avg_dc' : avg_dc}
        results_list.append(node)
        print(node)

    results_json = json.dumps(results_list, indent=2)

    with open(output_file, "w") as f:
        f.write(results_json)


    return results_json


avg_pair_sim(input_file, 'keepID')

{'platform': 'Tiktok', 'avg_js': 21.50336669463446, 'avg_dc': 26.37937115437659}
{'platform': 'Facebook', 'avg_js': 26.561450307060806, 'avg_dc': 32.02148355641113}
{'platform': 'Instagram', 'avg_js': 706.3068726020231, 'avg_dc': 856.4583909639834}
{'platform': 'Twitter', 'avg_js': 0.26975187463224337, 'avg_dc': 0.36965003668450963}
{'platform': 'Youtube', 'avg_js': 6.2743866028391215, 'avg_dc': 7.118806780701509}


'[\n  {\n    "platform": "Tiktok",\n    "avg_js": 21.50336669463446,\n    "avg_dc": 26.37937115437659\n  },\n  {\n    "platform": "Facebook",\n    "avg_js": 26.561450307060806,\n    "avg_dc": 32.02148355641113\n  },\n  {\n    "platform": "Instagram",\n    "avg_js": 706.3068726020231,\n    "avg_dc": 856.4583909639834\n  },\n  {\n    "platform": "Twitter",\n    "avg_js": 0.26975187463224337,\n    "avg_dc": 0.36965003668450963\n  },\n  {\n    "platform": "Youtube",\n    "avg_js": 6.2743866028391215,\n    "avg_dc": 7.118806780701509\n  }\n]'